In [1]:
import pandas as pd
import numpy as np
import datetime as dt


In [2]:
customers_raw = pd.read_csv("funnel_data/dirty/customers.csv")
products_raw = pd.read_csv("funnel_data/dirty/products.csv")
events_raw = pd.read_csv("funnel_data/dirty/events.csv")
orders_raw = pd.read_csv("funnel_data/dirty/orders.csv")
order_items_raw = pd.read_csv("funnel_data/dirty/order_items.csv")

raw_tables = {
    "customers": customers_raw,
    "products": products_raw,
    "events": events_raw,
    "orders": orders_raw,
    "order_items": order_items_raw,
}

In [3]:
for name, df in raw_tables.items():
    print(f"*****DATA PROFILE FOR {name}*****")
    print(f"Shape: {df.shape}")
    print("-------------------------------------")
    print(f"\nColumn data types:\n{df.dtypes}")
    print("-------------------------------------")
    print(f"\nNull counts:\n{df.isna().sum()}")
    print("-------------------------------------")
    print(f"\nDuplicates:\n{df.duplicated().sum()}")
    print("-------------------------------------\n")

    

*****DATA PROFILE FOR customers*****
Shape: (50000, 4)
-------------------------------------

Column data types:
customer_id           object
signup_date           object
acquisition_source    object
country               object
dtype: object
-------------------------------------

Null counts:
customer_id           0
signup_date           0
acquisition_source    0
country               0
dtype: int64
-------------------------------------

Duplicates:
0
-------------------------------------

*****DATA PROFILE FOR products*****
Shape: (500, 5)
-------------------------------------

Column data types:
product_id     object
category       object
subcategory    object
brand          object
price          object
dtype: object
-------------------------------------

Null counts:
product_id      0
category        0
subcategory     0
brand          15
price           1
dtype: int64
-------------------------------------

Duplicates:
0
-------------------------------------

*****DATA PROFILE FOR e

In [4]:
categorical_columns = {
    "customers": ["acquisition_source", "country"],
    "products": ["category", "subcategory", "brand"],
    "events": ["event_type", "device", "traffic_source", "category", "subcategory", "brand"],
    "orders": ["payment_method", "order_status"],
}

for table_name, columns in categorical_columns.items():
    df = raw_tables[table_name]

    for column in columns:
        print(f"\n--- {table_name}.{column} ---")
        print(df[column].value_counts(dropna=False).head(20))


--- customers.acquisition_source ---
acquisition_source
Paid Search       14019
Social            11506
Email              9048
Organic Search     6464
Referral           5923
Direct             3040
Name: count, dtype: int64

--- customers.country ---
country
US           22022
Canada        8431
UK            6095
Germany       5553
France        4395
Australia     3504
Name: count, dtype: int64

--- products.category ---
category
 apparel              154
Sportswear            105
Home and Lifestyle     98
accessories            74
FOOTWEAR               69
Name: count, dtype: int64

--- products.subcategory ---
subcategory
Decor            53
T-Shirts         51
Activewear       50
Pants            46
Sneakers         42
Cosmetics        34
Jackets          33
Bags             31
Training         28
Running          27
Hoodies          24
Belts            23
Hats             20
Sandals          15
Boots            12
Water Bottles    11
Name: count, dtype: int64

--- products.bran

In [5]:
numeric_columns = {
    "products": ["price"],
    "events": ["price"],
    "orders": ["total_amount"],
    "order_items": ["unit_price", "quantity", "discount_pct", "line_total"],
}

for table_name, columns in numeric_columns.items():
    df = raw_tables[table_name]

    for column in columns:
        print(f"\n--- {table_name}.{column} ---")
        print(df[column].describe())


--- products.price ---
count       499
unique      483
top       30.92
freq          3
Name: price, dtype: object

--- events.price ---
count    1.059422e+06
mean     3.835820e+01
std      2.495527e+01
min      8.380000e+00
25%      2.107000e+01
50%      3.242000e+01
75%      4.670000e+01
max      1.718200e+02
Name: price, dtype: float64

--- orders.total_amount ---
count     26052
unique     3811
top       30.92
freq         68
Name: total_amount, dtype: object

--- order_items.unit_price ---
count    26052.000000
mean        38.511423
std         25.087329
min          8.380000
25%         21.070000
50%         32.620000
75%         46.790000
max        171.820000
Name: unit_price, dtype: float64

--- order_items.quantity ---
count    26052.000000
mean         1.232151
std          0.490826
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000
Name: quantity, dtype: float64

--- order_items.discount_pct ---
count    26052.00000

In [6]:
date_columns = {
    "customers": ["signup_date"],
    "events": ["event_timestamp"],
    "orders": ["order_timestamp"],
}

for table_name, columns in date_columns.items():
    df = raw_tables[table_name]

    for column in columns:
        dates = pd.to_datetime(df[column], errors="coerce")

        print(f"\n--- {table_name}.{column} ---")
        print(f"Original dtype: {df[column].dtype}")
        print(f"Unparseable values: {dates.isna().sum()}")
        print(f"Earliest valid date: {dates.min()}")
        print(f"Latest valid date: {dates.max()}")


--- customers.signup_date ---
Original dtype: object
Unparseable values: 0
Earliest valid date: 2025-01-01 00:00:00
Latest valid date: 2026-05-31 00:00:00

--- events.event_timestamp ---
Original dtype: object
Unparseable values: 1
Earliest valid date: 2025-01-01 23:15:00.099934123
Latest valid date: 2026-06-02 00:22:59.053422419

--- orders.order_timestamp ---
Original dtype: object
Unparseable values: 0
Earliest valid date: 2025-01-01 23:19:00.299374104
Latest valid date: 2026-06-02 00:03:33.405892844


In [7]:
primary_keys = {
    "customers": "customer_id",
    "products": "product_id",
    "events": "event_id",
    "orders": "order_id",
    "order_items": "order_id"
}

for table_name, key in primary_keys.items():
    df = raw_tables[table_name]

    print(f"\n--- {table_name}.{key} ---")
    print(f"Missing IDs: {df[key].isna().sum():,}")
    print(f"Distinct IDs: {df[key].nunique():,}")
    print(f"Duplicate ID rows: {df[key].duplicated().sum():,}")


--- customers.customer_id ---
Missing IDs: 0
Distinct IDs: 50,000
Duplicate ID rows: 0

--- products.product_id ---
Missing IDs: 0
Distinct IDs: 500
Duplicate ID rows: 0

--- events.event_id ---
Missing IDs: 0
Distinct IDs: 1,058,364
Duplicate ID rows: 1,058

--- orders.order_id ---
Missing IDs: 0
Distinct IDs: 26,026
Duplicate ID rows: 26

--- order_items.order_id ---
Missing IDs: 0
Distinct IDs: 26,026
Duplicate ID rows: 26


In [8]:

relationships = [
    ("events", "customer_id", "customers", "customer_id"),
    ("events", "product_id", "products", "product_id"),
    ("orders", "customer_id", "customers", "customer_id"),
    ("orders", "session_id", "events", "session_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
]

for child_table, child_key, parent_table, parent_key in relationships:
    child = raw_tables[child_table]
    parent = raw_tables[parent_table]

    missing_matches = ~child[child_key].isin(parent[parent_key])

    print(
        f"{child_table}.{child_key} -> "
        f"{parent_table}.{parent_key}: "
        f"{missing_matches.sum():,} rows without a match"
    )

events.customer_id -> customers.customer_id: 0 rows without a match
events.product_id -> products.product_id: 0 rows without a match
orders.customer_id -> customers.customer_id: 0 rows without a match
orders.session_id -> events.session_id: 0 rows without a match
order_items.order_id -> orders.order_id: 0 rows without a match
order_items.product_id -> products.product_id: 0 rows without a match
